In [1]:
import os
os.chdir('/home/smallyan/eval_agent')
print(f"Working directory: {os.getcwd()}")

Working directory: /home/smallyan/eval_agent


In [2]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

CUDA available: True
GPU Device: NVIDIA A40


In [3]:
# Check the structure of the original repo
original_repo = '/net/scratch2/smallyan/InterpDetect_eval'
import os

# List root contents
print("=== Original Repo Root Contents ===")
for item in os.listdir(original_repo):
    print(f"  {item}")

# Look for documentation files
print("\n=== Looking for documentation files ===")
for root, dirs, files in os.walk(original_repo):
    for f in files:
        if 'documentation' in f.lower() or f.endswith('.md'):
            print(f"  {os.path.join(root, f)}")

=== Original Repo Root Contents ===
  doc_only_evaluation
  documentation.pdf
  plan.md
  .gitignore
  CodeWalkthrough.md
  trained_models
  LICENSE
  .git
  evaluation
  no_exe_evaluation
  scripts
  requirements.txt
  datasets

=== Looking for documentation files ===
  /net/scratch2/smallyan/InterpDetect_eval/documentation.pdf
  /net/scratch2/smallyan/InterpDetect_eval/plan.md
  /net/scratch2/smallyan/InterpDetect_eval/CodeWalkthrough.md
  /net/scratch2/smallyan/InterpDetect_eval/doc_only_evaluation/replication_evaluation.md


  /net/scratch2/smallyan/InterpDetect_eval/evaluation/replication_eval/documentation_eval_summary.json
  /net/scratch2/smallyan/InterpDetect_eval/evaluation/replication_eval/documentation_evaluation_summary.md
  /net/scratch2/smallyan/InterpDetect_eval/evaluation/replications/documentation_replication.md
  /net/scratch2/smallyan/InterpDetect_eval/evaluation/replications/evaluation_replication.md
  /net/scratch2/smallyan/InterpDetect_eval/no_exe_evaluation/replications/no_exe_evaluation_replication.md
  /net/scratch2/smallyan/InterpDetect_eval/scripts/preprocess/README.md


In [4]:
# Read the original documentation - start with CodeWalkthrough.md
with open(os.path.join(original_repo, 'CodeWalkthrough.md'), 'r') as f:
    original_walkthrough = f.read()
print("=== CodeWalkthrough.md ===")
print(original_walkthrough[:10000])

=== CodeWalkthrough.md ===
# InterpDetect

**InterpDetect: Interpretable Signals for Detecting Hallucinations in Retrieval-Augmented Generation**

[![Paper](https://img.shields.io/badge/Paper-OpenReview-blue)](https://openreview.net/pdf?id=TZzBKwHLwF)
[![License](https://img.shields.io/badge/License-MIT-green.svg)](LICENSE)

A comprehensive framework for detecting and analyzing hallucinations in Retrieval-Augmented Generation (RAG) systems using interpretability techniques and chunk-level analysis on the RAGBench/FinQA dataset.

## Overview

This project implements a novel approach to hallucination detection by leveraging interpretability methods. The framework consists of three main parts:

1. **Preprocessing Pipeline** - Convert raw datasets to labeled data with hallucination spans
2. **Training & Prediction** - Train classifiers or use pre-trained models for hallucination detection
3. **Baseline Comparisons** - Evaluate against multiple baseline methods (RAGAS, TruLens, RefChecker, 

In [5]:
# Read the plan.md file to understand the experiment
with open(os.path.join(original_repo, 'plan.md'), 'r') as f:
    plan_content = f.read()
print("=== plan.md ===")
print(plan_content)

=== plan.md ===
# Plan
## Objective
Develop a mechanistic interpretability-based hallucination detection method for Retrieval-Augmented Generation (RAG) systems by computing External Context Scores (ECS) across layers and attention heads and Parametric Knowledge Scores (PKS) across layers (FFN), training regression-based classifiers on these signals, and demonstrating generalization from a small proxy model (Qwen3-0.6b) to larger production models (GPT-4.1-mini).

## Hypothesis
1. RAG hallucinations correlate with:  later-layer FFN modules disproportionately inject parametric knowledge into the residual stream while attention heads fail to adequately exploit external context.
2. External Context Score (ECS) and Parametric Knowledge Score (PKS) are correlated with hallucination occurrence and can serve as predictive features for hallucination detection.
3. Mechanistic signals extracted from a small proxy model (0.6b parameters) can generalize to detect hallucinations in responses from l

In [6]:
# Now read the replicated documentation
replication_path = os.path.join(original_repo, 'evaluation/replications/documentation_replication.md')
with open(replication_path, 'r') as f:
    replicated_doc = f.read()
print("=== documentation_replication.md ===")
print(replicated_doc)

=== documentation_replication.md ===
# InterpDetect Replication Documentation

## Goal

Replicate the InterpDetect experiment for detecting hallucinations in Retrieval-Augmented Generation (RAG) systems using mechanistic interpretability signals (ECS and PKS).

## Data

### Source
- **Repository**: InterpDetect (https://openreview.net/pdf?id=TZzBKwHLwF)
- **Model**: Qwen3-0.6B (smallest available model as per replication guidelines)

### Training Data
- **Location**: `/datasets/train/train3000_w_chunk_score_part*.json` (18 files)
- **Samples**: 1800 responses → 7799 spans
- **After balancing**: 6786 spans (3393 hallucinated, 3393 non-hallucinated)
- **Split**: 90% train, 10% validation

### Test Data
- **Location**: `/datasets/test/test_w_chunk_score_qwen06b.json`
- **Samples**: 256 responses → 975 spans
- **Distribution**: 699 non-hallucinated, 276 hallucinated

### Features
- **ECS (External Context Score)**: 448 features (28 layers × 16 attention heads)
  - Measures cosine similarit

In [7]:
# Let me also check if there's any more detailed documentation in the evaluation folder
eval_replication_path = os.path.join(original_repo, 'evaluation/replications/evaluation_replication.md')
with open(eval_replication_path, 'r') as f:
    eval_replication = f.read()
print("=== evaluation_replication.md ===")
print(eval_replication)

=== evaluation_replication.md ===
# InterpDetect Replication Evaluation

## Reflection

This replication successfully reproduced the InterpDetect experiment for hallucination detection in RAG systems using mechanistic interpretability signals. The core methodology involving External Context Score (ECS) and Parametric Knowledge Score (PKS) was verified through correlation analysis, and the classifier training pipeline was replicated with results closely matching the original paper.

### What Worked Well
1. **Clear documentation**: The plan.md and CodeWalkthrough.md provided sufficient detail to understand the methodology
2. **Pre-computed scores**: The repository included pre-computed ECS/PKS scores, enabling focus on classifier training and evaluation
3. **Standard libraries**: The use of scikit-learn and standard ML libraries made replication straightforward
4. **Pre-trained models**: Availability of trained models allowed direct comparison

### Challenges Encountered
1. **Data format

In [8]:
# Now let's systematically compare the original vs replicated documentation

print("=" * 80)
print("COMPARISON: ORIGINAL vs REPLICATED DOCUMENTATION")
print("=" * 80)

print("\n### DE1. RESULT FIDELITY ANALYSIS ###\n")

# Original results from plan.md
original_results = {
    "ECS Correlation": "Negative (all attention heads)",
    "PKS Correlation": "Positive (later layers)",
    "SVC Validation F1": "76.60%",
    "Self-Eval Response F1": "74.68%",
}

# Replicated results from documentation_replication.md
replicated_results = {
    "ECS Correlation": "-0.2987 (negative)",
    "PKS Correlation": "+0.2768 (positive)",
    "SVC Validation F1": "76.01%",
    "Self-Eval Response F1": "73.42% (replicated) vs 74.68% (pre-trained)",
}

print("ORIGINAL RESULTS (from plan.md):")
for k, v in original_results.items():
    print(f"  - {k}: {v}")

print("\nREPLICATED RESULTS (from documentation_replication.md):")
for k, v in replicated_results.items():
    print(f"  - {k}: {v}")

print("\nDEVIATION ANALYSIS:")
# SVC F1 deviation
svc_original = 76.60
svc_replicated = 76.01
svc_deviation = abs(svc_original - svc_replicated) / svc_original * 100
print(f"  - SVC Validation F1 deviation: {svc_deviation:.2f}% (within 5% tolerance)")

# Response F1 deviation (pre-trained comparison)
resp_original = 74.68
resp_replicated = 73.42
resp_deviation = abs(resp_original - resp_replicated) / resp_original * 100
print(f"  - Response F1 deviation (replicated vs pre-trained): {resp_deviation:.2f}% (within 5% tolerance)")

COMPARISON: ORIGINAL vs REPLICATED DOCUMENTATION

### DE1. RESULT FIDELITY ANALYSIS ###

ORIGINAL RESULTS (from plan.md):
  - ECS Correlation: Negative (all attention heads)
  - PKS Correlation: Positive (later layers)
  - SVC Validation F1: 76.60%
  - Self-Eval Response F1: 74.68%

REPLICATED RESULTS (from documentation_replication.md):
  - ECS Correlation: -0.2987 (negative)
  - PKS Correlation: +0.2768 (positive)
  - SVC Validation F1: 76.01%
  - Self-Eval Response F1: 73.42% (replicated) vs 74.68% (pre-trained)

DEVIATION ANALYSIS:
  - SVC Validation F1 deviation: 0.77% (within 5% tolerance)
  - Response F1 deviation (replicated vs pre-trained): 1.69% (within 5% tolerance)


In [9]:
print("\n### DE2. CONCLUSION CONSISTENCY ANALYSIS ###\n")

print("ORIGINAL CONCLUSIONS (from plan.md):")
original_conclusions = [
    "1. RAG hallucinations correlate with later-layer FFN modules injecting parametric knowledge while attention heads fail to exploit external context",
    "2. ECS and PKS are correlated with hallucination occurrence and can serve as predictive features",
    "3. Mechanistic signals from small proxy model (0.6b) can generalize to detect hallucinations in larger models",
    "4. All attention heads exhibit negative correlations; hallucinated responses utilize less external context",
    "5. Later-layer FFNs exhibit higher PKS for hallucinated responses and are positively correlated with hallucinations",
    "6. SVC achieved highest validation F1 (76.60%); XGBoost overfitted",
    "7. Method achieved F1=74.68%, outperforming TruLens and llama-3.1-8b-instant, comparable to RefChecker"
]
for c in original_conclusions:
    print(f"  {c}")

print("\n\nREPLICATED CONCLUSIONS (from documentation_replication.md):")
replicated_conclusions = [
    "1. ECS negatively correlates with hallucination (-0.2987), supporting hypothesis that hallucinated responses use less external context",
    "2. PKS positively correlates with hallucination (+0.2768), supporting hypothesis of parametric knowledge injection",
    "3. Later layers (20-25) show strongest PKS correlation, consistent with paper's later-layer claim",
    "4. SVC achieves best validation F1 (76.01%), closely matching paper's 76.60%",
    "5. XGBoost shows severe overfitting (Train: 99.82%, Val: 74.82%), as noted in paper",
    "6. Core mechanistic hypothesis (ECS/PKS correlation) is verified",
    "7. Numerical results are within acceptable tolerance (<2% difference)"
]
for c in replicated_conclusions:
    print(f"  {c}")

print("\n\nCONCLUSION CONSISTENCY VERDICT: All major conclusions are consistent")
print("  - Direction of correlations matches (ECS negative, PKS positive)")
print("  - Classifier selection reasoning matches (SVC best, XGBoost overfits)")
print("  - Performance metrics within acceptable tolerance")
print("  - Later-layer effect confirmed")


### DE2. CONCLUSION CONSISTENCY ANALYSIS ###

ORIGINAL CONCLUSIONS (from plan.md):
  1. RAG hallucinations correlate with later-layer FFN modules injecting parametric knowledge while attention heads fail to exploit external context
  2. ECS and PKS are correlated with hallucination occurrence and can serve as predictive features
  3. Mechanistic signals from small proxy model (0.6b) can generalize to detect hallucinations in larger models
  4. All attention heads exhibit negative correlations; hallucinated responses utilize less external context
  5. Later-layer FFNs exhibit higher PKS for hallucinated responses and are positively correlated with hallucinations
  6. SVC achieved highest validation F1 (76.60%); XGBoost overfitted
  7. Method achieved F1=74.68%, outperforming TruLens and llama-3.1-8b-instant, comparable to RefChecker


REPLICATED CONCLUSIONS (from documentation_replication.md):
  1. ECS negatively correlates with hallucination (-0.2987), supporting hypothesis that hallu

In [10]:
print("\n### DE3. NO EXTERNAL OR HALLUCINATED INFORMATION ###\n")

print("Checking replicated documentation for external/hallucinated information...")
print()

# Check for external references or information not in original
print("SOURCES REFERENCED IN REPLICATED DOCUMENTATION:")
sources = [
    "- OpenReview paper link (https://openreview.net/pdf?id=TZzBKwHLwF) - Present in original CodeWalkthrough.md",
    "- Qwen3-0.6B model - Mentioned in original plan.md",
    "- TransformerLens library - Mentioned in original plan.md",
    "- scikit-learn classifiers - Referenced in original documentation",
    "- Pre-computed scores from /datasets/ - Present in repository",
]
for s in sources:
    print(f"  {s}")

print("\n\nCHECKING FOR UNSUPPORTED CLAIMS:")
external_claims = [
    "BGE embeddings for ECS computation - Referenced but aligns with cosine similarity methodology",
    "hook_resid_mid, hook_resid_post - Technical details consistent with TransformerLens usage",
    "StandardScaler normalization - Standard preprocessing, consistent with methodology",
]
for c in external_claims:
    print(f"  {c}")

print("\n\nVERDICT: No external or hallucinated information detected")
print("  - All data sources reference the original repository")
print("  - All methods described align with original plan.md")
print("  - Technical details are consistent with stated libraries (TransformerLens, scikit-learn)")
print("  - No invented findings or external references introduced")


### DE3. NO EXTERNAL OR HALLUCINATED INFORMATION ###

Checking replicated documentation for external/hallucinated information...

SOURCES REFERENCED IN REPLICATED DOCUMENTATION:
  - OpenReview paper link (https://openreview.net/pdf?id=TZzBKwHLwF) - Present in original CodeWalkthrough.md
  - Qwen3-0.6B model - Mentioned in original plan.md
  - TransformerLens library - Mentioned in original plan.md
  - scikit-learn classifiers - Referenced in original documentation
  - Pre-computed scores from /datasets/ - Present in repository


CHECKING FOR UNSUPPORTED CLAIMS:
  BGE embeddings for ECS computation - Referenced but aligns with cosine similarity methodology
  hook_resid_mid, hook_resid_post - Technical details consistent with TransformerLens usage
  StandardScaler normalization - Standard preprocessing, consistent with methodology


VERDICT: No external or hallucinated information detected
  - All data sources reference the original repository
  - All methods described align with origin

In [11]:
# Create output directory
output_dir = os.path.join(original_repo, 'evaluation/new_replication_eval')
os.makedirs(output_dir, exist_ok=True)
print(f"Created output directory: {output_dir}")

Created output directory: /net/scratch2/smallyan/InterpDetect_eval/evaluation/new_replication_eval


In [12]:
# Write the documentation_evaluation_summary.md
documentation_summary_md = """# Documentation Evaluation Summary

## InterpDetect Replication - Documentation Evaluation

### Results Comparison

The replicated documentation faithfully reproduces the key results from the original experiment:

1. **ECS Correlation**: The replication reports a negative correlation (r = -0.2987, p < 0.001) between ECS and hallucination, consistent with the original finding that "all attention heads exhibit negative correlations" and "hallucinated responses utilize less external context."

2. **PKS Correlation**: The replication reports a positive correlation (r = +0.2768, p < 0.001) between PKS and hallucination, consistent with the original finding that "later-layer FFNs exhibit substantially higher PKS for hallucinated responses."

3. **Classifier Performance**: 
   - SVC Validation F1: 76.01% (replicated) vs 76.60% (original) — **0.77% deviation**
   - Response-level F1: 73.42% (replicated) vs 74.68% (pre-trained) — **1.69% deviation**
   - Both deviations are well within the 5% tolerance threshold

4. **Classifier Behavior**: XGBoost overfitting (Train: 99.82%, Val: 74.82%) matches the original observation that "XGBoost overfitted despite strong training performance."

### Conclusions Comparison

The replicated documentation presents conclusions that are fully consistent with the original:

1. **Core Hypothesis Verified**: The mechanistic hypothesis that hallucinations correlate with (a) reduced external context utilization (ECS) and (b) increased parametric knowledge injection (PKS) is confirmed.

2. **Later-Layer Effect**: The replication confirms that layers 20-25 show the strongest PKS correlation with hallucination, consistent with the original claim about later FFN layers.

3. **Classifier Selection**: SVC is identified as the best-performing classifier in both original and replication, with consistent reasoning (XGBoost overfits, SVC generalizes better).

4. **Performance Claims**: The response-level F1 performance (~74%) is consistent with the original claim of outperforming baselines like TruLens (67.32%) and llama-3.1-8b-instant (57.53%).

### External/Hallucinated Information

No external or hallucinated information was introduced in the replicated documentation:

- All data sources reference the original repository (`/datasets/train/`, `/datasets/test/`)
- All methods described align with the original plan.md (ECS, PKS, classifier training)
- Technical details (TransformerLens hooks, BGE embeddings, StandardScaler) are consistent with stated methodology
- No invented findings, external references, or unsupported claims were identified

---

## Documentation Evaluation Checklist

| Criterion | Status | Rationale |
|-----------|--------|-----------|
| **DE1. Result Fidelity** | **PASS** | All key metrics (ECS/PKS correlations, SVC F1, Response F1) match within 5% tolerance. Correlation directions (ECS negative, PKS positive) are consistent. |
| **DE2. Conclusion Consistency** | **PASS** | All conclusions align with the original: mechanistic hypothesis verified, later-layer effect confirmed, classifier selection reasoning matches. |
| **DE3. No External/Hallucinated Information** | **PASS** | No external references or invented findings introduced. All claims are grounded in the original documentation and repository data. |

---

## Final Verdict

**PASS**

The replicated documentation faithfully reproduces the results and conclusions of the original InterpDetect experiment. All numerical results are within the 5% deviation tolerance, all conclusions are consistent, and no external or hallucinated information was introduced.
"""

# Write to file
md_path = os.path.join(output_dir, 'documentation_evaluation_summary.md')
with open(md_path, 'w') as f:
    f.write(documentation_summary_md)
print(f"Written: {md_path}")

Written: /net/scratch2/smallyan/InterpDetect_eval/evaluation/new_replication_eval/documentation_evaluation_summary.md


In [13]:
import json

# Write the documentation_eval_summary.json
documentation_eval_json = {
    "Checklist": {
        "DE1_ResultFidelity": "PASS",
        "DE2_ConclusionConsistency": "PASS",
        "DE3_NoExternalInformation": "PASS"
    },
    "Rationale": {
        "DE1_ResultFidelity": "All key results match within 5% tolerance: SVC Validation F1 (76.01% vs 76.60%, 0.77% deviation), Response-level F1 (73.42% vs 74.68%, 1.69% deviation). Correlation directions are consistent (ECS negative r=-0.30, PKS positive r=+0.28). Classifier behavior patterns (XGBoost overfitting, SVC generalizing best) match the original findings.",
        "DE2_ConclusionConsistency": "All conclusions are consistent with the original: (1) Core mechanistic hypothesis verified - ECS negatively correlates and PKS positively correlates with hallucination; (2) Later-layer FFN effect (layers 20-25) confirmed; (3) SVC selected as best classifier matching original; (4) Performance claims align with baseline comparisons.",
        "DE3_NoExternalInformation": "No external or hallucinated information introduced. All data sources reference original repository paths. Technical details (TransformerLens hooks, BGE embeddings, StandardScaler) are consistent with stated methodology in plan.md and CodeWalkthrough.md. No invented findings or unsupported claims identified."
    }
}

# Write to file
json_path = os.path.join(output_dir, 'documentation_eval_summary.json')
with open(json_path, 'w') as f:
    json.dump(documentation_eval_json, f, indent=2)
print(f"Written: {json_path}")

# Display the JSON
print("\nJSON Content:")
print(json.dumps(documentation_eval_json, indent=2))

Written: /net/scratch2/smallyan/InterpDetect_eval/evaluation/new_replication_eval/documentation_eval_summary.json

JSON Content:
{
  "Checklist": {
    "DE1_ResultFidelity": "PASS",
    "DE2_ConclusionConsistency": "PASS",
    "DE3_NoExternalInformation": "PASS"
  },
  "Rationale": {
    "DE1_ResultFidelity": "All key results match within 5% tolerance: SVC Validation F1 (76.01% vs 76.60%, 0.77% deviation), Response-level F1 (73.42% vs 74.68%, 1.69% deviation). Correlation directions are consistent (ECS negative r=-0.30, PKS positive r=+0.28). Classifier behavior patterns (XGBoost overfitting, SVC generalizing best) match the original findings.",
    "DE2_ConclusionConsistency": "All conclusions are consistent with the original: (1) Core mechanistic hypothesis verified - ECS negatively correlates and PKS positively correlates with hallucination; (2) Later-layer FFN effect (layers 20-25) confirmed; (3) SVC selected as best classifier matching original; (4) Performance claims align with b

In [14]:
# Verify the output files
print("=== FINAL VERIFICATION ===\n")

print("Output directory contents:")
for item in os.listdir(output_dir):
    full_path = os.path.join(output_dir, item)
    size = os.path.getsize(full_path)
    print(f"  {item} ({size} bytes)")

print("\n" + "=" * 60)
print("DOCUMENTATION EVALUATION COMPLETE")
print("=" * 60)
print(f"\nOutput location: {output_dir}")
print("\nFiles created:")
print("  1. documentation_evaluation_summary.md")
print("  2. documentation_eval_summary.json")
print("\nFinal Verdict: PASS")
print("  - DE1. Result Fidelity: PASS")
print("  - DE2. Conclusion Consistency: PASS")
print("  - DE3. No External Information: PASS")

=== FINAL VERIFICATION ===

Output directory contents:
  documentation_evaluation_summary.md (3609 bytes)
  documentation_eval_summary.json (1243 bytes)

DOCUMENTATION EVALUATION COMPLETE

Output location: /net/scratch2/smallyan/InterpDetect_eval/evaluation/new_replication_eval

Files created:
  1. documentation_evaluation_summary.md
  2. documentation_eval_summary.json

Final Verdict: PASS
  - DE1. Result Fidelity: PASS
  - DE2. Conclusion Consistency: PASS
  - DE3. No External Information: PASS


# Documentation Evaluation Summary

## Task Completed

Evaluated the replicator's documentation against the original InterpDetect experiment documentation.

## Results

| Criterion | Status |
|-----------|--------|
| DE1. Result Fidelity | **PASS** |
| DE2. Conclusion Consistency | **PASS** |
| DE3. No External Information | **PASS** |

**Final Verdict: PASS**

## Output Files

- `/net/scratch2/smallyan/InterpDetect_eval/evaluation/new_replication_eval/documentation_evaluation_summary.md`
- `/net/scratch2/smallyan/InterpDetect_eval/evaluation/new_replication_eval/documentation_eval_summary.json`